# Reach-to-Grasp Preprocessing and Loader Validation for CTM

## 1. Purpose and Dataset Conventions

This notebook documents the preprocessing pipeline used to convert the reach-to-grasp dataset into a form suitable for Continuous Thought Machine experiments.

The goal is not only to produce a working loader, but also to keep the preprocessing assumptions explicit and reproducible.

The notebook therefore distinguishes between:

- information stored directly in the released dataset,
- behaviour reproduced from the original preprocessing source code,
- derived metadata introduced by the current loader,
- and later choices made specifically for CTM experiments.

The source CSV files are treated as immutable.

Corrections, class assignments, board-sensor mappings, movement boundaries, and diagnostic flags are stored only as derived information.

### 1.1 Dataset

The dataset contains recordings from five right-handed participants performing reach-to-grasp movements toward eight objects:

| Class | Object |
|---:|---|
| 0 | Stick |
| 1 | Ball |
| 2 | Screw |
| 3 | Box |
| 4 | Knife |
| 5 | Glass |
| 6 | Assembly |
| 7 | Disk |

The motion-capture glove contains 25 tracked landmarks with three coordinates each:

$$
25 \times 3 = 75
$$

kinematic dimensions.

For the final CTM representation, only the five fingertip landmarks are retained:

    SensorID_4
    SensorID_9
    SensorID_14
    SensorID_19
    SensorID_24

This gives:

$$
5 \times 3 = 15
$$

kinematic input channels.

The full 75-dimensional glove representation is still used where required to reproduce the original movement-boundary logic.

### 1.2 Experimental blocks

The recordings are organised into five blocks:

    B0, B1, B2, B3, B4

B0 contains the calibration or model-fitting trials.

B1-B4 contain the later experimental trials.

The class-label and board-sensor conventions differ between B0 and B1-B4, so these blocks cannot be treated identically during preprocessing.

### 1.3 Processed recording structure

Each processed trial CSV contains several signal groups.

The glove kinematics are stored as:

    SensorID_0_x
    SensorID_0_y
    SensorID_0_z
    ...
    SensorID_24_x
    SensorID_24_y
    SensorID_24_z

The object-board measurements are stored as eight channels:

    board_data_0
    board_data_1
    board_data_2
    board_data_3
    board_data_4
    board_data_5
    board_data_6
    board_data_7

Additional signals and metadata include:

    palm_data
    glove_timestamps
    board_timestamps
    palm_timestamps
    hand_active
    block
    trial_number
    rep_number
    attempt_number
    target_object
    trial_success

The palm signal is retained as a diagnostic signal, but it is not assumed to define the final movement boundary.

### 1.4 Relevant project paths

The current project uses:

    Project root:
    /Users/arthu/Desktop/CTM_Res-Pro

    Processed reach-to-grasp data:
    /Users/arthu/Desktop/CTM_Res-Pro/data/reach_to_grasp_dataset/data

    Loader:
    /Users/arthu/Desktop/CTM_Res-Pro/models/ctm/data_loader.py

    Preprocessing notebook:
    /Users/arthu/Desktop/CTM_Res-Pro/models/ctm/02_reach_to_grasp_preprocessing.ipynb

The preprocessing pipeline developed in this notebook will later provide fixed-format trajectories for the CTM experiments.

The current objective is therefore:

    reconstruct and validate preprocessing first
                |
                v
         build CTM samples
                |
                v
        run small CTM pilot
                |
                v
      run larger experiment grid

## 2. Inventory and Successful-Attempt Selection

Before applying any movement preprocessing, the loader first determines which physical CSV files correspond to retained experimental recordings.

### 2.1 Physical files versus logical recording slots

The processed dataset contains:

$$
976
$$

physical recording files.

A logical recording slot is defined by:

    participant
    block
    trial_number
    rep_number

Grouping the files by these identifiers produces:

$$
960
$$

logical recording slots.

The difference is caused by 16 slots that contain a failed attempt followed by a successful retry.

In every one of these retry cases:

    attempt 0 -> trial_success = 0
    attempt 1 -> trial_success = 1

The loader therefore does not assume that `attempt0` is automatically the recording that should be retained.

### 2.2 Successful-attempt selection

For each logical recording slot, the loader selects the unique recording satisfying:

    trial_success == 1

The selection function also checks for ambiguous cases.

Its intended behaviour is:

    one successful attempt  -> retain it
    zero successful attempts -> exclude the slot
    multiple successful attempts -> raise an error

Across the complete dataset, there are 18 unsuccessful physical recordings.

Sixteen belong to retry slots and have successful replacements.

Two B0 slots have no successful replacement:

    participant 2, B0, trial 4, rep 0
    participant 3, B0, trial 3, rep 1

These two recordings are also explicitly marked unsuccessful in the original preprocessing source code.

The final retained dataset therefore contains:

$$
958
$$

successful logical recordings.

### 2.3 Retained dataset structure

The retained counts by participant are:

| Participant | Recordings |
|---:|---:|
| 1 | 192 |
| 2 | 191 |
| 3 | 191 |
| 4 | 192 |
| 5 | 192 |

The retained counts by block are:

| Block | Recordings |
|---:|---:|
| B0 | 318 |
| B1 | 160 |
| B2 | 160 |
| B3 | 160 |
| B4 | 160 |

This establishes the sample set used by all later preprocessing stages.

The important distinction is:

    976 physical files
          |
          v
    960 logical recording slots
          |
          v
    958 retained successful recordings

In [1]:
from pathlib import Path
import pandas as pd

from data_loader import (
    build_recording_inventory,
    extract_raw_metadata,
    select_successful_recordings,
)


# Build the physical-file inventory
inventory = build_recording_inventory()

# Extract raw metadata once for every physical recording
raw_records = []

for _, row in inventory.iterrows():
    raw_records.append(
        extract_raw_metadata(
            Path(row["path"])
        )
    )

raw_metadata_df = pd.DataFrame(raw_records)


# Select the unique successful recording for each logical slot
selected_df, excluded_df = select_successful_recordings(
    raw_metadata_df
)


# Count logical recording slots before success filtering
slot_columns = [
    "participant",
    "block",
    "trial_number",
    "rep_number",
]

n_logical_slots = (
    raw_metadata_df[
        slot_columns
    ]
    .drop_duplicates()
    .shape[0]
)


print("Physical recording files:", len(raw_metadata_df))
print("Logical recording slots:", n_logical_slots)
print("Retained successful recordings:", len(selected_df))
print("Excluded logical slots:", len(excluded_df))


print("\nExcluded slots:")
display(excluded_df)


print("\nRetained recordings per participant:")
display(
    selected_df[
        "participant"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)


print("\nRetained recordings per block:")
display(
    selected_df[
        "block"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)

Physical recording files: 976
Logical recording slots: 960
Retained successful recordings: 958
Excluded logical slots: 2

Excluded slots:


,participant,block,trial_number,rep_number,n_attempts,files
0,2,0,4,0,1,[B0_trial4_rep0_attempt0.csv]
1,3,0,3,1,1,[B0_trial3_rep1_attempt0.csv]



Retained recordings per participant:


,n_recordings
participant,
1,192
2,191
3,191
4,192
5,192



Retained recordings per block:


,n_recordings
block,
0,318
1,160
2,160
3,160
4,160


## 3. Target Correction and Class / Board Mapping

After successful recordings have been selected, the loader resolves the target metadata and derives the experimental class label and board-sensor channel.

These are treated as separate operations because `target_object`, `trial_number`, and physical board location do not have the same meaning in every block.

### 3.1 Final valid target value

The field `target_object` is not always constant within a recording.

Some processed CSV files contain more than one non-missing value, for example:

    [3, 0]

Across the 976 physical recordings, 139 files contain more than one distinct non-missing `target_object` value.

The loader therefore does not use the first row of `target_object`.

Instead, it records:

    target_object_final
    target_values_seen
    target_changed_within_recording

where `target_object_final` is the final valid, non-missing value in the recording.

The original evaluation code also uses the final target value when identifying the grasped object.

A previous diagnostic produced a count of 146 changing targets because transitions from missing values to populated values were also counted. The current count of 139 uses the stricter definition:

    more than one distinct non-missing target value

The two counts therefore describe different diagnostics rather than contradictory results.

### 3.2 Repetition-0 target correction

For B1-B4, repetition 0 follows an additional lookup rule provided by the lab supervisor.

For each combination of:

    participant
    block
    trial_number

the target for repetition 0 is obtained from the selected successful repetition-1 recording belonging to the same trial.

Conceptually:

    repetition 0
          |
          v
    find matching repetition 1
          |
          v
    use repetition 1 final valid target

This is a lookup operation, not an arithmetic transformation.

The loader therefore does not apply rules such as:

    target + 1

or:

    target - 1

The source CSV remains unchanged.

Instead, the derived metadata preserve both:

    target_object_raw
    target_object_corrected

together with information about whether the lookup rule was applied and which recording was used as the reference.

The correction applies to:

$$
5 \times 4 \times 8 = 160
$$

retained repetition-0 recordings in B1-B4.

In the current released dataset:

    correction lookups performed = 160
    numeric target values changed = 0

The rule is still kept explicitly because it records the intended experimental interpretation even when the stored final value already agrees with the reference repetition.

### 3.3 Block-specific class and board mapping

The experimental mapping differs between B0 and B1-B4.

For B0:

    class_label = trial_number
    board_column = board_data_3

For B1-B4:

    class_label = target_object_corrected
    board_column = board_data_{trial_number}

This means that object identity and physical board location must remain separate concepts.

For example, a B1-B4 recording with:

    trial_number = 1
    target_object_corrected = 6

is represented as:

    class_label = 6
    board_column = board_data_1

The loader therefore stores explicit provenance:

    class_label
    class_label_source
    board_sensor_index
    board_column
    board_sensor_source

### 3.4 Dataset-wide consistency

Applying this mapping to all 958 retained recordings produces:

| Class | Recordings |
|---:|---:|
| 0 | 120 |
| 1 | 120 |
| 2 | 120 |
| 3 | 119 |
| 4 | 119 |
| 5 | 120 |
| 6 | 120 |
| 7 | 120 |

The two missing samples are explained exactly by the two unreplaced B0 failures:

    participant 3, trial 3 -> class 3
    participant 2, trial 4 -> class 4

Each of B1-B4 contains exactly:

$$
20
$$

recordings per class.

The resulting balance provides an important validation that the reconstructed target, class, and board-sensor semantics are internally consistent.

In [2]:
from data_loader import (
    apply_rep0_target_correction,
    derive_experimental_mapping,
)


# Apply target correction and derive block-specific mapping
mapped_records = []

for _, row in selected_df.iterrows():
    metadata = row.to_dict()

    corrected = apply_rep0_target_correction(
        metadata,
        selected_df,
    )

    mapped = derive_experimental_mapping(
        corrected
    )

    mapped_records.append(mapped)

mapped_df = pd.DataFrame(mapped_records)


# ---------------------------------------------------------
# Correction validation
# ---------------------------------------------------------

rep0_corrected = mapped_df[
    mapped_df["target_correction_applied"]
]

print(
    "Correction lookups performed:",
    len(rep0_corrected),
)

print(
    "Numeric target values changed:",
    (
        rep0_corrected["target_object_raw"]
        != rep0_corrected["target_object_corrected"]
    ).sum(),
)


print(
    "\nCorrection lookups per block:"
)

display(
    rep0_corrected[
        "block"
    ]
    .value_counts()
    .sort_index()
    .rename("n_corrections")
    .to_frame()
)


# ---------------------------------------------------------
# Class-label validation
# ---------------------------------------------------------

print(
    "\nOverall class counts:"
)

display(
    mapped_df[
        "class_label"
    ]
    .value_counts()
    .sort_index()
    .rename("n_recordings")
    .to_frame()
)


print(
    "\nClass counts by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["class_label"],
    )
)


# ---------------------------------------------------------
# Board-sensor validation
# ---------------------------------------------------------

print(
    "\nBoard columns by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["board_column"],
    )
)


print(
    "\nClass-label source by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["class_label_source"],
    )
)


print(
    "\nBoard-sensor source by block:"
)

display(
    pd.crosstab(
        mapped_df["block"],
        mapped_df["board_sensor_source"],
    )
)

Correction lookups performed: 160
Numeric target values changed: 0

Correction lookups per block:


,n_corrections
block,
1,40
2,40
3,40
4,40



Overall class counts:


,n_recordings
class_label,
0,120
1,120
2,120
3,119
4,119
5,120
6,120
7,120



Class counts by block:


...

board_column,board_data_0,board_data_1,board_data_2,board_data_3,board_data_4,board_data_5,board_data_6,board_data_7
block,,,,,,,,
0,0,0,0,318,0,0,0,0
1,20,20,20,20,20,20,20,20
2,20,20,20,20,20,20,20,20
3,20,20,20,20,20,20,20,20
4,20,20,20,20,20,20,20,20



Class-label source by block:


class_label_source,target_object_corrected,trial_number
block,,
0,0,318
1,160,0
2,160,0
3,160,0
4,160,0



Board-sensor source by block:


board_sensor_source,fixed_B0_sensor_3,trial_number
block,,
0,318,0
1,0,160
2,0,160
3,0,160
4,0,160


## 4. B0 Extraction Definition and Validation

The B0 movement interval is reconstructed directly from the original preprocessing code.

The implementation deliberately preserves several unusual details of the original helper functions so that reproduction and later methodological critique remain separate.

### 4.1 Preprocessing helpers used by B0

The threshold-crossing helper searches for the first sample satisfying:

$$
|x(t)| > \theta
$$

and returns one index after that first exceedance.

Therefore, if the first crossing occurs at Python index 2, the helper returns:

$$
3.
$$

This `+1` behaviour is preserved intentionally because it is part of the original implementation.

The glove-speed profile is calculated from all 75 glove coordinates:

$$
25 \times 3 = 75
$$

using the temporal gradient and Euclidean norm:

$$
v(t)
=
\left\|
\frac{d\mathbf{x}(t)}{dt}
\right\|_2.
$$

The full 75-dimensional glove signal is used only for reproducing the movement-boundary logic.

The later CTM representation will retain only the 15 fingertip coordinates.

### 4.2 Board-sensor cleanup

Before boundary detection, the original preprocessing replaces the board-sensor saturation values:

    8191
    8190

using the highest available value that is not equal to the target saturation value.

An important implementation detail is that this replacement is applied to the complete eight-channel board array.

Therefore, the replacement maximum can come from a different board channel than the channel later used for B0 boundary detection.

This behaviour is reproduced intentionally because it matches the original source code.

### 4.3 B0 endpoint

B0 always uses:

    board_data_3

as the board signal.

After saturation cleanup, the relative board signal is:

$$
b(t)
=
\text{board\_data\_3}(t)
-
\text{board\_data\_3}(0).
$$

The endpoint threshold is:

$$
\theta_{\mathrm{end}}
=
0.2 \max_t b(t).
$$

The detected endpoint is then obtained using the original threshold-crossing helper.

Because the helper applies an absolute value internally, the effective crossing condition is:

$$
|b(t)| > \theta_{\mathrm{end}}.
$$

This distinction is important for some of the flagged recordings discussed later.

### 4.4 B0 movement start

The start threshold is computed from the glove-speed profile before the detected endpoint:

$$
\theta_{\mathrm{start}}
=
0.1
\max_{t < t_{\mathrm{end}}}
v(t).
$$

The movement start is the first threshold crossing under the same helper rule.

Therefore, the B0 start used by the movement-export script is based on glove speed.

It is not defined by:

    hand_active

and it is not defined by:

    palm_data

Those signals remain useful diagnostics, but they represent different timing concepts.

### 4.5 Detected endpoint versus exported endpoint

The original movement-export script retains 50 samples after the detected endpoint:

    glove_data[start:end + 50]

The loader therefore preserves both:

    end_idx
    export_end_idx

with:

$$
\text{export\_end\_idx}
=
\text{end\_idx} + 50.
$$

This distinction is important for later CTM experiments.

The detected movement interval and the exported trajectory do not represent the same temporal extent.

For the median B0 recording:

    detected movement length = 81 samples
    exported trajectory length = 131 samples

The added 50 samples therefore account for approximately:

$$
\frac{50}{131} \approx 38\%
$$

of the exported sample count.

Consequently, a later window described as the "last quarter of the normalized export" should not automatically be interpreted as the last quarter of the physical reach.

Part of that window may occur after the detected endpoint.

### 4.6 Dataset-wide B0 validation

The reconstructed B0 extraction was applied to all 318 retained B0 recordings.

The result was:

| Check | Result |
|---|---:|
| Retained B0 recordings | 318 |
| Start/end pairs computed | 318 |
| Extraction exceptions | 0 |
| Export intervals exceeding recording length | 0 |

All 318 recordings produced ordered indices satisfying:

$$
t_{\mathrm{start}} < t_{\mathrm{end}}.
$$

This establishes computational reproducibility of the extraction rule.

It does not establish that every detected interval is physically plausible.

The detected movement lengths have:

    median = 81 samples
    middle 50% = 72-91 samples
    full range = 8-247 samples

The exported trajectories have:

    median = 131 samples
    full range = 58-297 samples

Physical plausibility remains under review for the flagged outliers described in the next section.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from data_loader import find_b0_movement_bounds


# Keep only the retained B0 recordings
b0_df = selected_df[
    selected_df["block"] == 0
].copy()


b0_results = []
b0_errors = []


for _, row in b0_df.iterrows():

    path = Path(row["path"])

    try:
        df = pd.read_csv(path)

        bounds = find_b0_movement_bounds(
            df
        )

        movement_length = (
            bounds["end_idx"]
            - bounds["start_idx"]
        )

        export_length = (
            bounds["export_end_idx"]
            - bounds["start_idx"]
        )

        export_exceeds_recording = (
            bounds["export_end_idx"]
            > len(df)
        )

        b0_results.append(
            {
                "participant": row["participant"],
                "trial_number": row["trial_number"],
                "rep_number": row["rep_number"],
                "file": row["file"],
                "start_idx": bounds["start_idx"],
                "end_idx": bounds["end_idx"],
                "export_end_idx": bounds["export_end_idx"],
                "movement_length": movement_length,
                "export_length": export_length,
                "export_exceeds_recording": export_exceeds_recording,
            }
        )

    except Exception as exc:

        b0_errors.append(
            {
                "file": row["file"],
                "error": str(exc),
            }
        )


b0_results_df = pd.DataFrame(
    b0_results
)

b0_errors_df = pd.DataFrame(
    b0_errors
)


# ---------------------------------------------------------
# Basic extraction checks
# ---------------------------------------------------------

print(
    "Retained B0 recordings:",
    len(b0_df),
)

print(
    "Start/end pairs computed:",
    len(b0_results_df),
)

print(
    "Extraction exceptions:",
    len(b0_errors_df),
)

print(
    "Export intervals exceeding recording length:",
    b0_results_df[
        "export_exceeds_recording"
    ].sum(),
)


# ---------------------------------------------------------
# Ordering check
# ---------------------------------------------------------

ordered_bounds = (
    b0_results_df["start_idx"]
    < b0_results_df["end_idx"]
)

print(
    "Ordered start/end pairs:",
    ordered_bounds.sum(),
    "/",
    len(b0_results_df),
)


# ---------------------------------------------------------
# Movement-length summary
# ---------------------------------------------------------

print(
    "\nDetected movement-length summary:"
)

display(
    b0_results_df[
        "movement_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# Export-length summary
# ---------------------------------------------------------

print(
    "\nExported trajectory-length summary:"
)

display(
    b0_results_df[
        "export_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# Compact key statistics
# ---------------------------------------------------------

movement_q1 = b0_results_df[
    "movement_length"
].quantile(0.25)

movement_median = b0_results_df[
    "movement_length"
].median()

movement_q3 = b0_results_df[
    "movement_length"
].quantile(0.75)

movement_min = b0_results_df[
    "movement_length"
].min()

movement_max = b0_results_df[
    "movement_length"
].max()

export_median = b0_results_df[
    "export_length"
].median()

export_min = b0_results_df[
    "export_length"
].min()

export_max = b0_results_df[
    "export_length"
].max()


print(
    "\nCompact summary:"
)

print(
    f"Detected movement median: {movement_median:.0f} samples"
)

print(
    f"Detected movement middle 50%: "
    f"{movement_q1:.0f}-{movement_q3:.0f} samples"
)

print(
    f"Detected movement range: "
    f"{movement_min:.0f}-{movement_max:.0f} samples"
)

print(
    f"Exported trajectory median: "
    f"{export_median:.0f} samples"
)

print(
    f"Exported trajectory range: "
    f"{export_min:.0f}-{export_max:.0f} samples"
)


if not b0_errors_df.empty:

    print(
        "\nExtraction errors:"
    )

    display(
        b0_errors_df
    )

/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1317: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2) / (dx1 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1318: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1319: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in multiply
  out[tuple(slice1)] = a * f[tuple(slice2)] + b * f[tuple(slice3)] \
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in add
  out[tuple(slice1)]

Retained B0 recordings: 318
Start/end pairs computed: 318
Extraction exceptions: 0
Export intervals exceeding recording length: 0
Ordered start/end pairs: 318 / 318

Detected movement-length summary:


,samples
count,318.000000
mean,83.028302
std,20.516112
min,8.000000
25%,72.000000
50%,81.000000
75%,91.000000
max,247.000000



Exported trajectory-length summary:


,samples
count,318.000000
mean,133.028302
std,20.516112
min,58.000000
25%,122.000000
50%,131.000000
75%,141.000000
max,297.000000



Compact summary:
Detected movement median: 81 samples
Detected movement middle 50%: 72-91 samples
Detected movement range: 8-247 samples
Exported trajectory median: 131 samples
Exported trajectory range: 58-297 samples


## 5. Flagged Recordings and Methodological Limits

The B0 extraction is computationally reproducible across all 318 retained recordings, but this does not imply that every computed interval is physically meaningful.

Several recordings were therefore flagged for closer inspection rather than automatically excluded.

### 5.1 Flagged B0 recordings

The main boundary outliers identified so far are:

| Participant | Trial | Rep | Detected length | Main observation |
|---:|---:|---:|---:|---|
| 5 | 4 | 0 | 8 samples | Very small board threshold causes an early endpoint |
| 3 | 5 | 2 | 20 samples | Glove speed is already high at the beginning of the recording |
| 4 | 6 | 6 | 247 samples | Glove-speed onset occurs much earlier than `hand_active` |
| 2 | 1 | 7 | 217 samples | Clear early glove-speed increase relative to object lift |

These recordings are retained because their unusual boundaries can be explained by the original preprocessing rule.

No additional exclusion criterion has been introduced.

The current principle is:

    reproduce the original method
              |
              v
        identify unusual cases
              |
              v
      evaluate them explicitly
              |
              v
    modify preprocessing only if justified

### 5.2 Participant 5 short-interval case

The shortest detected movement occurs in:

    participant 5
    B0_trial4_rep0_attempt0.csv

with:

    start_idx = 1
    end_idx = 9
    movement_length = 8

For this recording, the B0 endpoint threshold is only:

$$
\theta_{\mathrm{end}} = 1.
$$

The original method computes the threshold from:

$$
0.2 \max b(t),
$$

while the threshold-crossing helper tests:

$$
|b(t)| > \theta_{\mathrm{end}}.
$$

This means that a small negative board fluctuation can trigger the endpoint when the positive maximum is very small.

The recording also has:

    hand_active index = 0

so the recording itself already begins in an unusual state.

This case is computationally reproducible, but its physical interpretation as a complete reach-to-grasp interval remains questionable.

### 5.3 Early glove-speed onset

For:

    participant 3
    B0_trial5_rep2_attempt0.csv

the detected interval is only 20 samples long.

Here, the board endpoint corresponds to a substantial board-signal increase, but the glove-speed profile is already large at the beginning of the recording.

The resulting start is:

    start_idx = 1

while:

    hand_active index = 20

This illustrates that the glove-speed criterion and `hand_active` represent different onset definitions.

The same distinction is visible in the long participant-4 recording:

    participant 4
    B0_trial6_rep6_attempt0.csv

where:

    start_idx = 62
    hand_active index = 257

The speed profile shows sustained movement around the detected start, so the early start does not appear to be caused by a single isolated numerical spike.

### 5.4 Duplicate glove timestamp

Exactly one retained B0 recording contains an adjacent duplicate glove timestamp:

    participant 5
    B0_trial2_rep0_attempt0.csv

This produces NumPy warnings during:

    np.gradient(data, timestamps, axis=0)

because a repeated timestamp implies:

$$
\Delta t = 0.
$$

However, the duplicate occurs after the detected B0 endpoint.

Therefore, it is outside the region used to calculate:

    speed[:end]

and does not affect the B0 start threshold for this recording.

The timestamp sequence is currently left unchanged so that the reproduced preprocessing remains faithful to the original implementation.

### 5.5 Exported trajectories are not identical to detected reaches

The original export retains 50 samples after the detected endpoint.

Therefore:

    detected movement interval

and:

    exported trajectory

must remain conceptually separate.

For the median B0 example:

    detected interval = 81 samples
    exported interval = 131 samples

Approximately 38% of the exported sample count therefore lies in the added 50-sample portion.

This becomes especially important for later CTM window experiments.

For example, the final 25% of a normalized exported trajectory should not automatically be described as:

    the final 25% of the reach

because part of that interval may occur after the detected board-based endpoint.

Window positions should therefore be described relative to the exported trajectory unless a separate physical-reach normalization is introduced.

### 5.6 Current resampling convention

The current `resample_trajectory()` function performs linear interpolation over normalized sample position.

Conceptually, an input trajectory with $N$ rows is assigned coordinates spanning:

$$
0 \rightarrow 1
$$

according to row position, and the trajectory is interpolated onto a new fixed grid.

This means that the current operation is best described as:

    linear interpolation over normalized sample index

rather than:

    interpolation over physical timestamps

The method implicitly assumes that adjacent input samples are approximately equally spaced.

This is a reasonable explicit preprocessing choice for constructing fixed-size CTM inputs, but it should not be confused with physical-time interpolation.

Before making claims about timing in milliseconds or fractions of physical movement duration, interpolation using the recorded glove timestamps should be considered separately.

### 5.7 Current interpretation

The current loader therefore distinguishes three different questions:

    Can the original preprocessing rule be reproduced?

    Does the rule produce ordered numerical boundaries?

    Are those boundaries physically meaningful for every recording?

For B0, the answers currently are:

    reproduction: yes
    ordered boundaries: 318 / 318
    physical plausibility: generally plausible, with flagged exceptions

The flagged recordings remain part of the dataset for now and should remain visible during later CTM validation rather than being silently removed.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

from data_loader import (
    DATA_ROOT,
    FINGERTIP_COLUMNS,
    resample_trajectory,
)


# ---------------------------------------------------------
# 1. Compact table of the four flagged B0 recordings
# ---------------------------------------------------------

flagged_cases = [
    (5, 4, 0),
    (3, 5, 2),
    (4, 6, 6),
    (2, 1, 7),
]

flagged_mask = pd.Series(
    False,
    index=b0_results_df.index,
)

for participant, trial, rep in flagged_cases:

    flagged_mask |= (
        (b0_results_df["participant"] == participant)
        & (b0_results_df["trial_number"] == trial)
        & (b0_results_df["rep_number"] == rep)
    )


flagged_b0_df = (
    b0_results_df[
        flagged_mask
    ]
    [
        [
            "participant",
            "trial_number",
            "rep_number",
            "file",
            "start_idx",
            "end_idx",
            "export_end_idx",
            "movement_length",
            "export_length",
        ]
    ]
    .sort_values(
        [
            "movement_length",
            "participant",
        ]
    )
)


print("Flagged B0 recordings:")
display(flagged_b0_df)


# ---------------------------------------------------------
# 2. Audit duplicate glove timestamps
# ---------------------------------------------------------

duplicate_records = []

for _, row in b0_results_df.iterrows():

    path = (
        DATA_ROOT
        / f"participant_{int(row['participant'])}"
        / "motion"
        / row["file"]
    )

    df = pd.read_csv(path)

    glove_time = df[
        "glove_timestamps"
    ].to_numpy(
        dtype=float
    )

    duplicate_indices = np.where(
        np.diff(glove_time) == 0
    )[0]

    if len(duplicate_indices) > 0:

        duplicate_before_end = np.sum(
            duplicate_indices
            < row["end_idx"] - 1
        )

        duplicate_records.append(
            {
                "participant": row["participant"],
                "trial_number": row["trial_number"],
                "rep_number": row["rep_number"],
                "file": row["file"],
                "n_duplicate_steps": len(
                    duplicate_indices
                ),
                "duplicate_indices": (
                    duplicate_indices.tolist()
                ),
                "duplicates_before_end": int(
                    duplicate_before_end
                ),
            }
        )


duplicate_df = pd.DataFrame(
    duplicate_records
)


print(
    "\nB0 recordings with duplicate glove timestamps:",
    len(duplicate_df),
)

if not duplicate_df.empty:
    display(duplicate_df)


# ---------------------------------------------------------
# 3. Crop the 15-channel exported fingertip trajectories
#    and resample each one to the fixed CTM length
# ---------------------------------------------------------

resampling_checks = []

for _, row in b0_results_df.iterrows():

    path = (
        DATA_ROOT
        / f"participant_{int(row['participant'])}"
        / "motion"
        / row["file"]
    )

    df = pd.read_csv(path)

    fingertip_data = df[
        FINGERTIP_COLUMNS
    ].to_numpy(
        dtype=np.float32
    )

    start = int(
        row["start_idx"]
    )

    export_end = int(
        row["export_end_idx"]
    )

    exported = fingertip_data[
        start:export_end
    ]

    resampled = resample_trajectory(
        exported
    )

    resampling_checks.append(
        {
            "participant": row["participant"],
            "trial_number": row["trial_number"],
            "rep_number": row["rep_number"],
            "file": row["file"],
            "input_length": len(exported),
            "input_finite": bool(
                np.isfinite(
                    exported
                ).all()
            ),
            "resampled_shape": (
                tuple(
                    resampled.shape
                )
            ),
            "resampled_finite": bool(
                np.isfinite(
                    resampled
                ).all()
            ),
        }
    )


resampling_df = pd.DataFrame(
    resampling_checks
)


# ---------------------------------------------------------
# 4. Validate all exported/resampled B0 trajectories
# ---------------------------------------------------------

print(
    "\nExported fingertip trajectories checked:",
    len(resampling_df),
)

print(
    "Finite exported trajectories:",
    resampling_df[
        "input_finite"
    ].sum(),
    "/",
    len(resampling_df),
)

print(
    "Finite resampled trajectories:",
    resampling_df[
        "resampled_finite"
    ].sum(),
    "/",
    len(resampling_df),
)


print(
    "\nResampled shapes:"
)

display(
    resampling_df[
        "resampled_shape"
    ]
    .value_counts()
    .rename(
        "n_recordings"
    )
    .to_frame()
)


print(
    "\nExported input-length summary:"
)

display(
    resampling_df[
        "input_length"
    ]
    .describe()
    .to_frame(
        name="samples"
    )
)


# ---------------------------------------------------------
# 5. Show failures if any exist
# ---------------------------------------------------------

invalid_resampled = resampling_df[
    ~resampling_df[
        "resampled_finite"
    ]
]

if not invalid_resampled.empty:

    print(
        "\nNon-finite resampled trajectories:"
    )

    display(
        invalid_resampled
    )

Flagged B0 recordings:


,participant,trial_number,rep_number,file,start_idx,end_idx,export_end_idx,movement_length,export_length
286,5,4,0,B0_trial4_rep0_attempt0.csv,1,9,59,8,58
168,3,5,2,B0_trial5_rep2_attempt0.csv,1,21,71,20,70
79,2,1,7,B0_trial1_rep7_attempt0.csv,92,309,359,217,267
244,4,6,6,B0_trial6_rep6_attempt0.csv,62,309,359,247,297



B0 recordings with duplicate glove timestamps: 1


,participant,trial_number,rep_number,file,n_duplicate_steps,duplicate_indices,duplicates_before_end
0,5,2,0,B0_trial2_rep0_attempt0.csv,1,[432],0



Exported fingertip trajectories checked: 318
Finite exported trajectories: 318 / 318
Finite resampled trajectories: 318 / 318

Resampled shapes:


,n_recordings
resampled_shape,
"(15, 100)",318



Exported input-length summary:


,samples
count,318.000000
mean,133.028302
std,20.516112
min,58.000000
25%,122.000000
50%,131.000000
75%,141.000000
max,297.000000


In [8]:
# ---------------------------------------------------------
# Metadata consistency audit
# ---------------------------------------------------------
#
# The inventory obtains block/trial/rep/attempt identifiers
# from the filename.
#
# extract_raw_metadata() obtains the same identifiers from
# the CSV contents.
#
# Before moving to B1-B4, I will verify that both representations
# agree for every physical recording.
# ---------------------------------------------------------

identifier_columns = [
    "block",
    "trial_number",
    "rep_number",
    "attempt_number",
]

consistency_records = []


for _, inventory_row in inventory.iterrows():

    path = Path(
        inventory_row["path"]
    )

    df = pd.read_csv(
        path,
        usecols=identifier_columns,
    )

    record = {
        "participant": inventory_row["participant"],
        "file": inventory_row["file"],
    }

    recording_consistent = True

    for column in identifier_columns:

        # Identifier parsed from the filename
        filename_value = int(
            inventory_row[column]
        )

        # All valid values appearing inside the CSV
        csv_values = (
            df[column]
            .dropna()
            .unique()
        )

        # The identifier should remain constant throughout
        # one recording.
        is_constant = (
            len(csv_values) == 1
        )

        if is_constant:
            csv_value = int(
                csv_values[0]
            )

            matches_filename = (
                csv_value
                == filename_value
            )

        else:
            csv_value = None
            matches_filename = False

        record[
            f"{column}_filename"
        ] = filename_value

        record[
            f"{column}_csv"
        ] = csv_value

        record[
            f"{column}_constant"
        ] = is_constant

        record[
            f"{column}_matches"
        ] = matches_filename

        if (
            not is_constant
            or not matches_filename
        ):
            recording_consistent = False

    record[
        "all_identifiers_consistent"
    ] = recording_consistent

    consistency_records.append(
        record
    )


metadata_consistency_df = pd.DataFrame(
    consistency_records
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "Physical recordings checked:",
    len(metadata_consistency_df),
)

print(
    "Fully consistent recordings:",
    metadata_consistency_df[
        "all_identifiers_consistent"
    ].sum(),
    "/",
    len(metadata_consistency_df),
)


for column in identifier_columns:

    print(
        f"\n{column}:"
    )

    print(
        "  constant within recording:",
        metadata_consistency_df[
            f"{column}_constant"
        ].sum(),
        "/",
        len(metadata_consistency_df),
    )

    print(
        "  matches filename:",
        metadata_consistency_df[
            f"{column}_matches"
        ].sum(),
        "/",
        len(metadata_consistency_df),
    )


# ---------------------------------------------------------
# Show any mismatches
# ---------------------------------------------------------

inconsistent_df = metadata_consistency_df[
    ~metadata_consistency_df[
        "all_identifiers_consistent"
    ]
]

print(
    "\nRecordings with identifier inconsistencies:",
    len(inconsistent_df),
)

if not inconsistent_df.empty:

    display(
        inconsistent_df
    )

Physical recordings checked: 976
Fully consistent recordings: 976 / 976

block:
  constant within recording: 976 / 976
  matches filename: 976 / 976

trial_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

rep_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

attempt_number:
  constant within recording: 976 / 976
  matches filename: 976 / 976

Recordings with identifier inconsistencies: 0


## 6. Final B1-B4 Reconstruction and Unified Loader Validation

The preprocessing pipeline is now complete enough to provide the first real reach-to-grasp dataset for CTM experiments.

Sections 2-5 establish the dataset inventory, successful-attempt selection, target correction, class and board mapping, and the B0 extraction path. This section records the selected B1-B4 source-defined movement crop, the CTM resampling convention, and the end-to-end validation of the unified loader.

### 6.1 B1-B4 movement crop

A concrete B1-B4 trajectory crop was reconstructed from `Fig17.py` in the original reach-to-grasp codebase.

For B1-B4, that source path uses:

    first hand_active == 1
              |
              v
        movement start
              |
              v
    board_data_{trial_number}
              |
              v
    zero-baseline at movement start
              |
              v
    first crossing of 10% of the
    maximum relative board displacement
              |
              v
          object lift

The extracted movement interval is:

    glove_data[start:end]

with:

    end = start + t_lift

where `t_lift` is the board-threshold crossing measured relative to the movement start.

Unlike B0, this B1-B4 crop does not retain an additional 50 samples after the detected endpoint.

The loader therefore uses two intentionally different block-specific movement crops:

| Component | B0 | B1-B4 |
|---|---|---|
| Movement start | 10% of maximum glove speed before endpoint | first `hand_active == 1` |
| Board channel | `board_data_3` | `board_data_{trial_number}` |
| Board threshold | 20% of relative maximum | 10% of relative maximum |
| Extra samples after endpoint | 50 | 0 |
| Final retained channels | 15 fingertip coordinates | 15 fingertip coordinates |

This should be described narrowly as **block-specific source-defined movement crops**. The B1-B4 rule above is a concrete crop used by `Fig17.py`; it should not be interpreted as evidence that every original analysis used one universal movement-extraction definition.

### 6.2 Boundary diagnostics

Before fixing the B1-B4 crop, several candidate movement-onset definitions were compared across the 640 retained B1-B4 recordings:

- first `hand_active == 1`
- palm displacement crossing 10% of its relative maximum
- palm displacement crossing 50 mm

The onset definitions are not numerically equivalent. The palm-derived events generally occur earlier than `hand_active`, with the largest disagreements concentrated in trial 3.

The object-lift endpoint was substantially more stable. Recomputing the 10%-of-maximum board crossing after zero-baselining from different candidate starts produced a median endpoint difference of zero samples.

These diagnostics were used to understand the data, not to introduce a new correction rule. The final B1-B4 loader therefore follows the selected `Fig17.py` crop based on `hand_active`.

### 6.3 Short B1-B4 trajectories

The reproduced B1-B4 crop contains a small but important lower tail of very short trajectories.

Across the 640 B1-B4 recordings:

    trajectories with <= 10 original samples = 11

Nine of these 11 recordings belong to:

    B2, trial 3, class 4

The shortest crops contain only two original samples. After resampling to 100 time points, such a trajectory becomes a dense interpolation between very few observations; the fixed tensor shape therefore does not imply that the sample contains rich temporal information.

These recordings are retained for the initial reproduction-based CTM pilot because they arise from the selected source-defined crop rather than from a newly introduced preprocessing error.

They should, however, be treated as a documented sensitivity issue. Before interpreting later partial-observation, noise, or robustness experiments, results should be compared with and without a clearly documented short-trajectory rule rather than silently deleting these samples.

### 6.4 B1-B4 validation

The implemented B1-B4 path was tested across all retained B1-B4 recordings.

Results:

    retained B1-B4 recordings = 640
    successful boundary extractions = 640
    extraction errors = 0
    raw fingertip channels = 15
    all raw trajectories finite = True
    all resampled trajectories finite = True
    final resampled shape = (15, 100)

The B1-B4 classes are exactly balanced:

| Class | Count |
|---:|---:|
| 0 | 80 |
| 1 | 80 |
| 2 | 80 |
| 3 | 80 |
| 4 | 80 |
| 5 | 80 |
| 6 | 80 |
| 7 | 80 |

The selected B1-B4 crop lengths have:

    mean = 60.94 samples
    median = 60 samples
    middle 50% = 51-71 samples
    range = 2-257 samples

### 6.5 CTM resampling convention

The movement crop and the fixed-length resampling step are separate choices.

In `Fig17.py`, the cropped trajectory is resampled through `scipy.signal.resample`.

The current CTM loader instead uses `resample_trajectory()`, which performs linear interpolation over normalized sample index.

These are different numerical operations.

Neither method uses the recorded glove timestamps as the interpolation coordinate, so the current CTM representation should be described as:

> **Block-specific source-defined movement crops, followed by linear sample-index resampling chosen for the CTM experiments.**

The current loader therefore does not claim to reproduce the complete original resampling pipeline.

This distinction is intentional. The linear sample-index convention is kept fixed for the first CTM pilot so that movement-crop reconstruction and resampling are not changed at the same time.

### 6.6 Unified loader

The final loader exposes one block-aware trajectory path:

    raw processed recording
              |
              v
       metadata mapping
              |
              v
        identify block
          /       \
         /         \
       B0          B1-B4
        |             |
        v             v
    B0 source     B1-B4 source
      crop          crop
         \           /
          \         /
           v       v
       15 fingertip channels
              |
              v
    CTM linear sample-index
          resampling
              |
              v
          (15, 100)

The complete retained dataset contains:

$$
958
$$

recordings and produces:

$$
X \in \mathbb{R}^{958 \times 15 \times 100}
$$

with:

$$
y \in \{0,\ldots,7\}^{958}.
$$

The retained recordings are distributed by block as:

| Block | Count |
|---|---:|
| B0 | 318 |
| B1 | 160 |
| B2 | 160 |
| B3 | 160 |
| B4 | 160 |

The final class distribution is:

| Class | Count |
|---:|---:|
| 0 | 120 |
| 1 | 120 |
| 2 | 120 |
| 3 | 119 |
| 4 | 119 |
| 5 | 120 |
| 6 | 120 |
| 7 | 120 |

The end-to-end builder validation gives:

    X shape = (958, 15, 100)
    y shape = (958,)
    metadata rows = 958
    excluded logical slots = 2
    X dtype = float32
    y dtype = int64
    all X finite = True

The extraction-method counts are:

    B0 speed/board20/+50 = 318
    B1-B4 hand_active/board10 = 640

### 6.7 Known warning and methodological limits

One retained B0 recording contains a duplicate glove timestamp. Because the B0 speed profile uses `np.gradient(data, timestamps, axis=0)`, this produces the known NumPy divide-by-zero warning.

The duplicate occurs after the detected B0 endpoint and therefore does not affect the `speed[:end]` region used to determine movement onset.

The final unified dataset nevertheless satisfies:

    all X finite = True

The warning is therefore documented rather than suppressed or used as an exclusion criterion.

A second limitation is that the exported physical extent is not identical across blocks. B0 includes 50 post-endpoint samples, whereas the selected B1-B4 crop ends at object lift.

Therefore, a later statement such as "50% observed" means 50% of the **exported trajectory representation**. It should not automatically be interpreted as the same physical fraction of the reach-to-grasp movement in B0 and B1-B4.

### 6.8 Final loader interface

The complete dataset can now be constructed through:

```python
from data_loader import build_reach_to_grasp_dataset

X, y, metadata_df, excluded_df = (
    build_reach_to_grasp_dataset()
)
```

The preprocessing pipeline should now remain fixed for the first CTM pilot unless that pilot reveals a concrete implementation or numerical problem.

### 6.9 Next experimental step

The next stage is a small CTM pilot on the real reach-to-grasp dataset.

The first run should be treated primarily as a pipeline smoke test: confirm that the CTM accepts the arrays, trains without numerical or shape failures, and produces sensible outputs.

Before interpreting accuracy as a generalization result, the evaluation split must be defined explicitly.

Different splits answer different questions:

- a held-out participant tests generalization to a person not seen during training,
- held-out repetitions test new repetitions from participants already represented in training,
- cross-block evaluation additionally mixes the documented B0 and B1-B4 crop conventions.

Any learned normalization or feature scaling must be fitted on the training partition only.

Once a meaningful evaluation split has been chosen, per-class results should be reported, with particular attention to the flagged short B1-B4 trajectories.

Only after the pilot and split definition should the larger experiment grid be resumed, including partial observation, window position, correlated noise, internal CTM ticks, and certainty behaviour.


In [5]:
# ---------------------------------------------------------
# Final unified loader validation
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from data_loader import build_reach_to_grasp_dataset


X, y, metadata_df, excluded_df = (
    build_reach_to_grasp_dataset()
)


# Core shape and numerical checks
assert X.shape == (958, 15, 100)
assert y.shape == (958,)
assert len(metadata_df) == 958
assert len(excluded_df) == 2
assert np.isfinite(X).all()


print("X shape:", X.shape)
print("y shape:", y.shape)
print("metadata rows:", len(metadata_df))
print("excluded logical slots:", len(excluded_df))
print("all X finite:", np.isfinite(X).all())

print("\nClass counts:")
print(
    pd.Series(y)
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRecordings by block:")
print(
    metadata_df["block"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nExtraction methods:")
print(
    metadata_df["extraction_method"]
    .value_counts()
    .to_string()
)


# Short B1-B4 trajectories are retained, but explicitly flagged.
short_b1_b4 = metadata_df[
    (metadata_df["block"].isin([1, 2, 3, 4]))
    & (metadata_df["trajectory_length"] <= 10)
].copy()

print(
    "\nB1-B4 trajectories with <= 10 original samples:",
    len(short_b1_b4),
)

if not short_b1_b4.empty:
    print("\nShort-trajectory concentration:")
    print(
        short_b1_b4
        .groupby(
            ["block", "trial_number", "class_label"]
        )
        .size()
        .sort_values(ascending=False)
        .to_string()
    )


/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1317: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2) / (dx1 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1318: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1319: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in multiply
  out[tuple(slice1)] = a * f[tuple(slice2)] + b * f[tuple(slice3)] \
/Users/arthu/miniconda3/envs/ctm_robotics/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:1328: RuntimeWarning: invalid value encountered in add
  out[tuple(slice1)]

X shape: (958, 15, 100)
y shape: (958,)
metadata rows: 958
excluded logical slots: 2
all X finite: True

Class counts:
0    120
1    120
2    120
3    119
4    119
5    120
6    120
7    120

Recordings by block:
block
0    318
1    160
2    160
3    160
4    160

Extraction methods:
extraction_method
B1-B4_hand_active_board10    640
B0_speed_board20_plus50      318

B1-B4 trajectories with <= 10 original samples: 11

Short-trajectory concentration:
block  trial_number  class_label
2      3             4              9
       1             6              1
4      3             7              1


# Appendix A. Historical Development and Rejected Assumptions

This appendix preserves the main exploratory steps that led to the current preprocessing pipeline.

These analyses are no longer part of the active loader specification, but they are retained because they explain why several apparently reasonable preprocessing assumptions were rejected.

## A.1 First-pass loader

The first version of the reach-to-grasp loader used a simplified interpretation of the processed recordings.

It approximately followed:

    read trial
        |
        v
    use hand_active as movement onset
        |
        v
    identify an object-board signal heuristically
        |
        v
    determine an endpoint
        |
        v
    crop fingertip trajectory
        |
        v
    resample to fixed length

A single-trial test produced the expected tensor shape, but participant-level evaluation revealed substantial problems.

For participant 1:

    physical processed files = 193
    successfully loaded = 183
    skipped failed recordings = 1
    loader errors = 9

The resulting class distribution was also strongly inconsistent with the intended balanced eight-class experiment.

This showed that producing correctly shaped arrays was not sufficient evidence that the dataset semantics had been interpreted correctly.

The loader was therefore decomposed into smaller metadata, selection, mapping, and boundary-detection stages.

## A.2 Investigating hand_active as movement onset

The first-pass loader used:

    hand_active == 1

as the movement-start criterion.

Because the dataset also contains a rest-position ToF signal, an exploratory alternative onset was defined using `palm_data`.

For participant 1, the first crossing of approximately 50 mm above the resting palm position was compared with the first `hand_active` sample.

Across all 192 successful participant-1 recordings, the palm-derived event occurred earlier than `hand_active`.

The observed delay had approximately:

    mean = 138.3 ms
    median = 112 ms
    minimum = 14 ms
    maximum = 722 ms

For the original validation recording:

    B0_trial0_rep1_attempt0.csv

the diagnostic timings were:

    palm +50 mm onset index = 229
    palm onset time = 3598 ms

    hand_active index = 239
    hand_active time = 3754 ms

    difference = 156 ms

This analysis demonstrated that `hand_active` should not automatically be interpreted as physical movement onset.

However, the palm +50 mm rule was itself only a diagnostic hypothesis.

Inspection of the original source code later showed that the B0 movement-export pipeline determines movement start from the glove-derived speed profile.

The palm analysis is therefore retained as historical evidence that multiple onset definitions exist, not as the final B0 preprocessing rule.

For B1-B4, later source reconstruction identified `Fig17.py` as the explicit reach-to-grasp trajectory path. That path uses the first `hand_active == 1` sample as movement start, so the final B1-B4 loader preserves that source-defined rule despite the diagnostic differences between candidate onset definitions.

## A.3 Discovering changes within target_object

Early loader versions treated `target_object` as if it were constant throughout each recording.

Inspection of the processed CSV files showed that this is not always true.

Some recordings contain sequences such as:

    [3, 0]

or other changes during the recording.

This explained why reading only the first row could produce misleading metadata.

The current loader instead records:

    target_object_final
    target_values_seen
    target_changed_within_recording

and uses the final valid non-missing target where required.

A diagnostic count found 139 recordings containing more than one distinct non-missing target value.

An earlier count of 146 also included transitions from missing values to populated values.

The two counts therefore correspond to different definitions.

## A.4 Rejected direct target-to-board mapping

An early hypothesis was that object identity might directly determine the physical board channel:

    target_object -> board_data_{target_object}

This was tested empirically on 192 successful participant-1 recordings.

The board channel associated with `target_object` was compared with the channel showing the strongest response.

Exact agreement occurred in only:

$$
\frac{88}{192}
\approx 45.8\%.
$$

The observed rank distribution was:

    rank 1 -> 88
    rank 2 -> 14
    rank 3 -> 17
    rank 4 -> 14
    rank 5 -> 18
    rank 6 -> 15
    rank 7 -> 13
    rank 8 -> 13

This was far too weak to support a direct object-to-board mapping.

The hypothesis was therefore rejected.

Subsequent source-code reconstruction and experimental clarification established the block-specific rules now used in the main preprocessing pipeline:

    B0:
        class label = trial_number
        board sensor = board_data_3

    B1-B4:
        class label = corrected target_object
        board sensor = board_data_{trial_number}

## A.5 Why this history is retained

These exploratory steps illustrate an important principle of the project:

    plausible interpretation
          |
          v
    empirical diagnostic
          |
          v
    compare with source code
          |
          v
    retain or reject assumption

The current loader should therefore be understood as the result of progressively resolving dataset semantics rather than as a direct translation of the first preprocessing idea.